Imports

In [ ]:
from pathlib import Path
import json, csv
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from PIL import Image

from sklearn.metrics import precision_recall_fscore_support, roc_auc_score, confusion_matrix

# =========================
# CONFIG
# =========================
PROCESSED_ROOT = Path("../../data/processed/SHAMPOOBLADEINTRAY_COMPLETEV2/gray")
INDEX_CSV = PROCESSED_ROOT / "index.csv"
TEST_LABELS_JSON = Path("../../data/labels/SHAMPOOBLADEINTRAY_COMPLETEV2/gray/test.json")

MODEL_PATH = Path("../../models/classifier/SHAMPOOBLADEINTRAY_COMPLETEV2/gray_multihead_stable/checkpoints/train_best.pt")

IMAGE_SIZE = 1024
BATCH_SIZE = 16

GRAY_MEAN = (0.5,)
GRAY_STD = (0.25,)

SPATIAL_CLASSES = ["isolated", "overlap"]          # 0, 1
THREAT_CLASSES = ["non_contraband", "contraband"] # 0, 1

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# =========================
# MODEL
# =========================
class SimpleCNN_MultiHead(nn.Module):
    def __init__(self, in_channels=1):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 16, 3, padding=1), nn.BatchNorm2d(16), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(256, 512, 3, padding=1), nn.BatchNorm2d(512), nn.ReLU(), nn.MaxPool2d(2),
        )

        self.gap = nn.AdaptiveAvgPool2d((1, 1))

        self.shared_fc = nn.Sequential(
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
        )

        self.spatial_head = nn.Linear(512, 2)
        self.threat_head = nn.Linear(512, 2)

    def forward(self, x):
        x = self.features(x)
        x = self.gap(x)
        x = x.view(x.size(0), -1)
        x = self.shared_fc(x)

        spatial_logits = self.spatial_head(x)
        threat_logits = self.threat_head(x)

        return spatial_logits, threat_logits

# =========================
# DATA
# =========================
def read_index_csv(path):
    rows = []
    with open(path, "r", newline="") as f:
        reader = csv.DictReader(f)
        for r in reader:
            rows.append(r)
    return rows

def load_label_map(label_path):
    data = json.load(open(label_path, "r"))
    label_map = {}

    for item in data:
        image_path = item["image"].replace("\\", "/")
        fname = Path(image_path).name

        overlap = int(item.get("overlap", 0))
        isolated = int(item.get("isolated", 0))
        contraband = int(item.get("contraband", 0))
        non_contraband = int(item.get("non_contraband", 0))

        # spatial: 0=isolated, 1=overlap
        if overlap == 1 and isolated == 0:
            spatial = 1
        elif overlap == 0 and isolated == 1:
            spatial = 0
        else:
            raise ValueError(f"Bad spatial label for {image_path}: overlap={overlap}, isolated={isolated}")

        # threat: 0=non_contraband, 1=contraband
        if contraband == 1 and non_contraband == 0:
            threat = 1
        elif contraband == 0 and non_contraband == 1:
            threat = 0
        else:
            raise ValueError(f"Bad threat label for {image_path}: contraband={contraband}, non_contraband={non_contraband}")

        label_map[image_path] = {"spatial": spatial, "threat": threat}
        label_map[fname] = {"spatial": spatial, "threat": threat}

    return label_map

class MultiHeadEvalDataset(Dataset):
    def __init__(self, index_rows, processed_root, split, label_map, transform=None):
        self.processed_root = processed_root
        self.transform = transform
        self.filepaths = []
        self.spatial_labels = []
        self.threat_labels = []

        for r in index_rows:
            fp = r["filepath"].replace("\\", "/")
            row_split = r["split"].strip().lower()

            if row_split != split:
                continue

            fname = Path(fp).name
            label = label_map.get(fp, label_map.get(fname, None))

            if label is None:
                raise KeyError(f"Missing label for {fp}")

            self.filepaths.append(fp)
            self.spatial_labels.append(label["spatial"])
            self.threat_labels.append(label["threat"])

        print(f"Loaded {len(self.filepaths)} {split} samples")

    def __len__(self):
        return len(self.filepaths)

    def __getitem__(self, idx):
        rel_path = self.filepaths[idx]
        img_path = self.processed_root / rel_path

        img = Image.open(img_path).convert("L")

        if self.transform:
            img = self.transform(img)

        spatial_y = torch.tensor(self.spatial_labels[idx], dtype=torch.long)
        threat_y = torch.tensor(self.threat_labels[idx], dtype=torch.long)

        return img, spatial_y, threat_y, rel_path

# IMPORTANT: match training transform
eval_transform = T.Compose([
    T.Resize(int(IMAGE_SIZE * 1.10)),
    T.CenterCrop(IMAGE_SIZE),
    T.ToTensor(),
    T.Normalize(GRAY_MEAN, GRAY_STD),
])

index_rows = read_index_csv(INDEX_CSV)
label_map = load_label_map(TEST_LABELS_JSON)

test_ds = MultiHeadEvalDataset(
    index_rows=index_rows,
    processed_root=PROCESSED_ROOT,
    split="test",
    label_map=label_map,
    transform=eval_transform,
)

test_loader = DataLoader(
    test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
)

# =========================
# LOAD MODEL SAFELY
# =========================
model = SimpleCNN_MultiHead(in_channels=1).to(device)

try:
    state = torch.load(MODEL_PATH, map_location=device, weights_only=True)
except TypeError:
    state = torch.load(MODEL_PATH, map_location=device)

if isinstance(state, dict):
    if "model_state" in state:
        state = state["model_state"]
    elif "model_state_dict" in state:
        state = state["model_state_dict"]

model.load_state_dict(state, strict=True)
model.eval()

print("Loaded model:", MODEL_PATH)

# =========================
# EVALUATE
# =========================
all_paths = []

spatial_true = []
spatial_pred = []
spatial_prob = []

threat_true = []
threat_pred = []
threat_prob = []

with torch.no_grad():
    for imgs, spatial_y, threat_y, paths in test_loader:
        imgs = imgs.to(device)

        spatial_logits, threat_logits = model(imgs)

        spatial_probs = torch.softmax(spatial_logits, dim=1)
        threat_probs = torch.softmax(threat_logits, dim=1)

        spatial_preds = spatial_probs.argmax(dim=1)
        threat_preds = threat_probs.argmax(dim=1)

        spatial_true.extend(spatial_y.numpy().tolist())
        spatial_pred.extend(spatial_preds.cpu().numpy().tolist())
        spatial_prob.extend(spatial_probs.cpu().numpy().tolist())

        threat_true.extend(threat_y.numpy().tolist())
        threat_pred.extend(threat_preds.cpu().numpy().tolist())
        threat_prob.extend(threat_probs.cpu().numpy().tolist())

        all_paths.extend(paths)

spatial_true = np.array(spatial_true)
spatial_pred = np.array(spatial_pred)
spatial_prob = np.array(spatial_prob)

threat_true = np.array(threat_true)
threat_pred = np.array(threat_pred)
threat_prob = np.array(threat_prob)

both_correct = (spatial_true == spatial_pred) & (threat_true == threat_pred)

# =========================
# METRICS
# =========================
def print_head_metrics(name, y_true, y_pred, y_prob, class_names):
    acc = (y_true == y_pred).mean()

    p, r, f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="binary",
        zero_division=0,
    )

    try:
        auc = roc_auc_score(y_true, y_prob[:, 1])
    except Exception:
        auc = 0.0

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])

    print(f"\n===== {name.upper()} HEAD =====")
    print(f"Class 0: {class_names[0]}")
    print(f"Class 1: {class_names[1]}")
    print(f"Accuracy : {acc:.4f}")
    print(f"Precision: {p:.4f}")
    print(f"Recall   : {r:.4f}")
    print(f"F1       : {f1:.4f}")
    print(f"AUC      : {auc:.4f}")
    print("Confusion Matrix:")
    print(cm)

    return {
        "accuracy": acc,
        "precision": p,
        "recall": r,
        "f1": f1,
        "auc": auc,
        "cm": cm,
    }

print("\n===== OVERALL RESULT =====")
print(f"Total samples : {len(all_paths)}")
print(f"Both correct  : {both_correct.sum()}")
print(f"Both wrong    : {(~both_correct).sum()}")
print(f"Both accuracy : {both_correct.mean():.4f}")

spatial_metrics = print_head_metrics(
    "spatial",
    spatial_true,
    spatial_pred,
    spatial_prob,
    SPATIAL_CLASSES,
)

threat_metrics = print_head_metrics(
    "threat",
    threat_true,
    threat_pred,
    threat_prob,
    THREAT_CLASSES,
)

# =========================
# RESULTS DATAFRAME
# =========================
results_df = pd.DataFrame({
    "image": all_paths,
    "spatial_true_id": spatial_true,
    "spatial_pred_id": spatial_pred,
    "spatial_true": [SPATIAL_CLASSES[i] for i in spatial_true],
    "spatial_pred": [SPATIAL_CLASSES[i] for i in spatial_pred],
    "spatial_prob_isolated": spatial_prob[:, 0],
    "spatial_prob_overlap": spatial_prob[:, 1],

    "threat_true_id": threat_true,
    "threat_pred_id": threat_pred,
    "threat_true": [THREAT_CLASSES[i] for i in threat_true],
    "threat_pred": [THREAT_CLASSES[i] for i in threat_pred],
    "threat_prob_non_contraband": threat_prob[:, 0],
    "threat_prob_contraband": threat_prob[:, 1],

    "both_correct": both_correct,
})

correct_indices = np.where(both_correct)[0]
wrong_indices = np.where(~both_correct)[0]

print("\n===== SAMPLE PREDICTIONS =====")
display(results_df.head(10))



In [ ]:
# =========================
# PART 2 — MULTI-HEAD GRAPHS + CORRECT/WRONG EXAMPLES
# =========================

# =========================
# SAFETY CHECK
# =========================
required_vars = [
    "test_ds",
    "PROCESSED_ROOT",
    "SPATIAL_CLASSES",
    "THREAT_CLASSES",
    "spatial_true",
    "spatial_pred",
    "spatial_prob",
    "threat_true",
    "threat_pred",
    "threat_prob",
    "both_correct",
]

missing = [v for v in required_vars if v not in globals()]

if missing:
    raise RuntimeError(
        f"Missing variables: {missing}. Run Part 1 multi-head evaluation first."
    )

correct_indices = np.where(both_correct)[0]
wrong_indices = np.where(~both_correct)[0]

print("Correct samples:", len(correct_indices))
print("Wrong samples:", len(wrong_indices))

# =========================
# OVERALL CORRECT VS WRONG GRAPH
# =========================
correct_count = int(both_correct.sum())
wrong_count = int((~both_correct).sum())

plt.figure(figsize=(5, 4))
plt.bar(["Correct", "Wrong"], [correct_count, wrong_count])
plt.title("Both-Heads Correct vs Wrong Predictions")
plt.ylabel("Number of images")
plt.show()

# =========================
# HEAD ACCURACY GRAPH
# =========================
spatial_acc = (spatial_true == spatial_pred).mean()
threat_acc = (threat_true == threat_pred).mean()
both_acc = both_correct.mean()

plt.figure(figsize=(7, 4))
plt.bar(
    ["Spatial", "Threat", "Both"],
    [spatial_acc, threat_acc, both_acc],
)
plt.title("Multi-Head Accuracy")
plt.ylabel("Accuracy")
plt.ylim(0, 1)
plt.show()

# =========================
# HEAD F1 GRAPH
# =========================
spatial_p, spatial_r, spatial_f1, _ = precision_recall_fscore_support(
    spatial_true,
    spatial_pred,
    average="binary",
    zero_division=0,
)

threat_p, threat_r, threat_f1, _ = precision_recall_fscore_support(
    threat_true,
    threat_pred,
    average="binary",
    zero_division=0,
)

plt.figure(figsize=(6, 4))
plt.bar(
    ["Spatial F1", "Threat F1"],
    [spatial_f1, threat_f1],
)
plt.title("F1 Score Per Head")
plt.ylabel("F1")
plt.ylim(0, 1)
plt.show()

# =========================
# TRUE VS PRED COUNT GRAPH
# =========================
spatial_true_counts = np.bincount(spatial_true, minlength=2)
spatial_pred_counts = np.bincount(spatial_pred, minlength=2)

threat_true_counts = np.bincount(threat_true, minlength=2)
threat_pred_counts = np.bincount(threat_pred, minlength=2)

labels = [
    "isolated_true",
    "overlap_true",
    "non_contraband_true",
    "contraband_true",
]

true_counts = [
    spatial_true_counts[0],
    spatial_true_counts[1],
    threat_true_counts[0],
    threat_true_counts[1],
]

pred_counts = [
    spatial_pred_counts[0],
    spatial_pred_counts[1],
    threat_pred_counts[0],
    threat_pred_counts[1],
]

x = np.arange(len(labels))

plt.figure(figsize=(11, 5))
plt.bar(x - 0.2, true_counts, width=0.4, label="True count")
plt.bar(x + 0.2, pred_counts, width=0.4, label="Pred count")
plt.xticks(x, labels, rotation=45)
plt.title("True vs Predicted Count Per Class")
plt.ylabel("Count")
plt.legend()
plt.show()

# =========================
# CONFUSION MATRIX HELPER
# =========================
def plot_confusion_matrix(cm, title, class_names):
    plt.figure(figsize=(4, 4))
    plt.imshow(cm)
    plt.title(title)
    plt.xticks([0, 1], [f"Pred {class_names[0]}", f"Pred {class_names[1]}"], rotation=30)
    plt.yticks([0, 1], [f"True {class_names[0]}", f"True {class_names[1]}"])

    for yy in range(2):
        for xx in range(2):
            plt.text(xx, yy, cm[yy, xx], ha="center", va="center")

    plt.colorbar()
    plt.tight_layout()
    plt.show()

spatial_cm = confusion_matrix(spatial_true, spatial_pred, labels=[0, 1])
threat_cm = confusion_matrix(threat_true, threat_pred, labels=[0, 1])

plot_confusion_matrix(
    spatial_cm,
    "Spatial Confusion Matrix",
    SPATIAL_CLASSES,
)

plot_confusion_matrix(
    threat_cm,
    "Threat Confusion Matrix",
    THREAT_CLASSES,
)

# =========================
# IMAGE EXAMPLES
# =========================
def show_prediction_example(dataset, index):
    img_tensor, spatial_y, threat_y, rel_path = dataset[int(index)]

    original_path = PROCESSED_ROOT / rel_path
    original_img = Image.open(original_path).convert("L")

    plt.figure(figsize=(7, 5))
    plt.imshow(original_img, cmap="gray", vmin=0, vmax=255)
    plt.axis("off")
    plt.title("Correct" if both_correct[index] else "Wrong")
    plt.show()

    print("Image:", rel_path)
    print("Both correct:", bool(both_correct[index]))

    print("\nSpatial head:")
    print("  True:", SPATIAL_CLASSES[int(spatial_true[index])])
    print("  Pred:", SPATIAL_CLASSES[int(spatial_pred[index])])
    print(f"  Prob isolated: {spatial_prob[index, 0]:.4f}")
    print(f"  Prob overlap : {spatial_prob[index, 1]:.4f}")

    print("\nThreat head:")
    print("  True:", THREAT_CLASSES[int(threat_true[index])])
    print("  Pred:", THREAT_CLASSES[int(threat_pred[index])])
    print(f"  Prob non-contraband: {threat_prob[index, 0]:.4f}")
    print(f"  Prob contraband    : {threat_prob[index, 1]:.4f}")

def show_many_prediction_examples(dataset, indices, title, max_images=5):
    indices = list(indices)

    print("\n" + "=" * 80)
    print(title)
    print("=" * 80)

    if len(indices) == 0:
        print("No examples found.")
        return

    for idx in indices[:max_images]:
        show_prediction_example(dataset, int(idx))

show_many_prediction_examples(
    dataset=test_ds,
    indices=correct_indices,
    title="Correct Prediction Examples",
    max_images=5,
)

show_many_prediction_examples(
    dataset=test_ds,
    indices=wrong_indices,
    title="Wrong Prediction Examples",
    max_images=5,
)

# =========================
# LABEL-SPECIFIC WRONG EXAMPLES
# =========================
def show_wrong_examples_for_head(head_name, max_images=5):
    if head_name == "spatial":
        wrong_for_head = np.where(spatial_true != spatial_pred)[0]
        title = "Wrong Examples for Spatial Head"

    elif head_name == "threat":
        wrong_for_head = np.where(threat_true != threat_pred)[0]
        title = "Wrong Examples for Threat Head"

    else:
        raise ValueError("head_name must be 'spatial' or 'threat'")

    show_many_prediction_examples(
        dataset=test_ds,
        indices=wrong_for_head,
        title=title,
        max_images=max_images,
    )

show_wrong_examples_for_head("spatial", max_images=5)
show_wrong_examples_for_head("threat", max_images=5)

In [ ]:
# =========================
# SAFETY CHECK
# =========================
required_vars = [
    "model", "test_ds", "LABEL_COLUMNS", "NUM_CLASSES",
    "y_true", "y_pred", "y_prob", "exact_match",
    "correct_indices", "wrong_indices"
]

missing = [v for v in required_vars if v not in globals()]

if missing:
    raise RuntimeError(
        f"Missing variables: {missing}. Run Part 1 first."
    )

# =========================
# GRADCAM
# =========================
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None

        self.fwd_handle = target_layer.register_forward_hook(self.forward_hook)
        self.bwd_handle = target_layer.register_full_backward_hook(self.backward_hook)

    def forward_hook(self, module, input, output):
        self.activations = output.detach()

    def backward_hook(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach()

    def generate(self, input_tensor, target_class):
        self.model.zero_grad(set_to_none=True)

        logits = self.model(input_tensor)
        score = logits[:, target_class].sum()
        score.backward()

        gradients = self.gradients
        activations = self.activations

        weights = gradients.mean(dim=(2, 3), keepdim=True)
        cam = (weights * activations).sum(dim=1, keepdim=True)
        cam = torch.relu(cam)

        cam = torch.nn.functional.interpolate(
            cam,
            size=input_tensor.shape[2:],
            mode="bilinear",
            align_corners=False,
        )

        cam = cam.squeeze().cpu().numpy()
        cam = cam - cam.min()
        cam = cam / (cam.max() + 1e-8)

        return cam

    def remove_hooks(self):
        self.fwd_handle.remove()
        self.bwd_handle.remove()

def denormalize_gray(tensor_img):
    img = tensor_img.squeeze(0).cpu().numpy()
    img = img * GRAY_STD[0] + GRAY_MEAN[0]
    img = np.clip(img, 0, 1)
    return img

def overlay_gradcam(gray_img, cam):
    gray_uint8 = np.uint8(gray_img * 255)
    heatmap = np.uint8(255 * cam)

    heatmap_color = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)
    gray_color = cv2.cvtColor(gray_uint8, cv2.COLOR_GRAY2BGR)

    overlay = cv2.addWeighted(gray_color, 0.55, heatmap_color, 0.45, 0)
    overlay = cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB)

    return overlay

def get_conv_layers(model):
    conv_layers = []

    for idx, layer in enumerate(model.features):
        if isinstance(layer, nn.Conv2d):
            conv_layers.append((f"features[{idx}]", layer))

    return conv_layers

conv_layers = get_conv_layers(model)

print("Detected Conv layers:")
for name, layer in conv_layers:
    print(name, layer)

def show_gradcam_each_layer(dataset, index=0, target_label_name="contraband"):
    model.eval()

    if target_label_name not in LABEL_COLUMNS:
        raise ValueError(f"Unknown label: {target_label_name}")

    target_class = LABEL_COLUMNS.index(target_label_name)

    img_tensor, label_tensor, rel_path = dataset[index]
    input_tensor = img_tensor.unsqueeze(0).to(device)

    with torch.no_grad():
        logits = model(input_tensor)
        probs = torch.sigmoid(logits).cpu().numpy()[0]
        preds = (probs >= THRESHOLD).astype(int)

    gray_img = denormalize_gray(img_tensor)

    true_labels = [LABEL_COLUMNS[i] for i in range(NUM_CLASSES) if label_tensor[i].item() == 1]
    pred_labels = [LABEL_COLUMNS[i] for i in range(NUM_CLASSES) if preds[i] == 1]

    print("\nImage:", rel_path)
    print("Target Grad-CAM label:", target_label_name)
    print("True labels:", true_labels)
    print("Pred labels:", pred_labels)

    n_layers = len(conv_layers)
    plt.figure(figsize=(4 * (n_layers + 1), 4))

    plt.subplot(1, n_layers + 1, 1)
    plt.imshow(gray_img, cmap="gray", vmin=0, vmax=1)
    plt.title("Input")
    plt.axis("off")

    for j, (layer_name, layer) in enumerate(conv_layers):
        gradcam = GradCAM(model, layer)
        cam = gradcam.generate(input_tensor, target_class)
        gradcam.remove_hooks()

        overlay = overlay_gradcam(gray_img, cam)

        plt.subplot(1, n_layers + 1, j + 2)
        plt.imshow(overlay)
        plt.title(layer_name)
        plt.axis("off")

    plt.tight_layout()
    plt.show()

    print("\nProbabilities:")
    for i, name in enumerate(LABEL_COLUMNS):
        print(f"  {name}: {probs[i]:.4f}")

def show_gradcam_multiple_examples(dataset, indices, target_label_name="contraband", max_images=3):
    indices = list(indices)

    if len(indices) == 0:
        print("No examples found.")
        return

    for idx in indices[:max_images]:
        show_gradcam_each_layer(
            dataset=dataset,
            index=int(idx),
            target_label_name=target_label_name,
        )

print("Correct examples:", correct_indices[:5])
print("Wrong examples:", wrong_indices[:5])

print("\nGrad-CAM correct examples:")
show_gradcam_multiple_examples(test_ds, correct_indices, target_label_name="contraband", max_images=3)

print("\nGrad-CAM wrong examples:")
show_gradcam_multiple_examples(test_ds, wrong_indices, target_label_name="contraband", max_images=3)

In [ ]:
from pathlib import Path
import json, csv, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
from collections import defaultdict

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.functional as TF
from PIL import Image, ImageDraw

from sklearn.metrics import (
    precision_recall_fscore_support,
    roc_auc_score,
    confusion_matrix,
    classification_report,
)

# =========================
# CONFIG
# =========================
PROCESSED_ROOT = Path("../../data/processed/SHAMPOOBLADEINTRAY_COMPLETEV2/gray")
INDEX_CSV = PROCESSED_ROOT / "index.csv"

LABEL_STUDIO_JSON = Path("../../data/raw/SHAMPOOBLADEINTRAY_COMPLETEV2/result_labels.json")
TEST_LABELS_JSON = Path("../../data/labels/SHAMPOOBLADEINTRAY_COMPLETEV2/gray/test.json")

#MODEL_PATH = Path("../../models/classifier/SHAMPOOBLADEINTRAY_COMPLETEV2/gray_multihead_itemmask/checkpoints/train_best.pt")
MODEL_PATH = Path("../../models/classifier/SHAMPOOBLADEINTRAY_COMPLETEV2/gray_multihead_itemmask_optional_slow/checkpoints/train_best_checkpoint.pt")


OUT_DIR = Path("../../reports/eval/SHAMPOOBLADEINTRAY_COMPLETEV2/gray_multihead_itemmask")
OUT_DIR.mkdir(parents=True, exist_ok=True)

IMAGE_SIZE = 512
BATCH_SIZE = 16
NUM_WORKERS = 2

GRAY_MEAN = (0.5,)
GRAY_STD = (0.25,)

MASK_MEAN = (0.5,)
MASK_STD = (0.5,)

# Evaluation must match training validation preprocessing: keep ratio + pad, no zoom/crop.

SPATIAL_CLASSES = ["isolated", "overlap"]          # 0, 1
THREAT_CLASSES = ["non_contraband", "contraband"] # 0, 1

ITEM_POLYGON_LABELS = {"shampoo", "blade"}        # tray excluded

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# =========================
# BASIC HELPERS
# =========================
def norm_label(x):
    return str(x).lower().strip().replace("-", "_").replace(" ", "_")


def base_group_id(filename: str) -> str:
    """
    Convert augmented names back to the original base name.

    Example:
      abc_orig.png     -> abc
      abc_balflip1.png -> abc
      abc_balflip2.png -> abc
      abc.png          -> abc
    """
    m = re.match(r"(.+?)_(orig|balflip\d+)\.[^.]+$", filename)
    if m:
        return m.group(1)
    return Path(filename).stem


def is_balflip(filename: str) -> bool:
    return re.search(r"_balflip\d+\.[^.]+$", filename) is not None


def extract_filename_from_task(item):
    if "image" in item:
        return Path(item.get("image", "")).name

    img_ref = item.get("data", {}).get("image", "")
    return Path(img_ref).name


# =========================
# LOAD RAW LABEL STUDIO ITEM POLYGONS
# =========================
def load_item_polygons_from_label_studio(json_path: Path):
    """
    Loads only item polygons: Shampoo + Blade.
    Tray polygons are ignored intentionally.
    """
    data = json.load(open(json_path, "r"))

    polygons_by_filename = defaultdict(list)

    for item in data:
        if not isinstance(item, dict):
            continue

        fname = extract_filename_from_task(item)
        if not fname:
            continue

        for ann in item.get("annotations", []):
            for res in ann.get("result", []):
                if res.get("type", "").lower() not in {"polygonlabels", "polygon"}:
                    continue

                value = res.get("value", {})
                poly_labels = value.get("polygonlabels", []) or value.get("labels", [])
                points = value.get("points", [])

                if not points:
                    continue

                for lab in poly_labels:
                    lab_norm = norm_label(lab)

                    if lab_norm not in ITEM_POLYGON_LABELS:
                        continue

                    polygons_by_filename[fname].append({
                        "label": lab_norm,
                        "points": points,
                    })

    print(f"Loaded item polygon masks for {len(polygons_by_filename)} original images")
    return polygons_by_filename


item_polygons_by_filename = load_item_polygons_from_label_studio(LABEL_STUDIO_JSON)


def find_original_polygon_key(processed_filename: str):
    suffix = Path(processed_filename).suffix
    original_name = base_group_id(processed_filename) + suffix

    if processed_filename in item_polygons_by_filename:
        return processed_filename

    if original_name in item_polygons_by_filename:
        return original_name

    return None


def create_item_only_mask(processed_filename: str, image_size):
    """
    Builds an item-only binary mask from raw Label Studio polygons.
    Only Shampoo + Blade are drawn.
    Tray is excluded.

    For files named *_balflipX.png, polygon x coordinates are horizontally flipped.
    """
    w, h = image_size
    mask = Image.new("L", (w, h), 0)
    draw = ImageDraw.Draw(mask)

    polygon_key = find_original_polygon_key(processed_filename)

    if polygon_key is None:
        return mask

    polygons = item_polygons_by_filename.get(polygon_key, [])
    flip_x = is_balflip(processed_filename)

    for poly in polygons:
        pts = poly["points"]

        xy = []
        for x_pct, y_pct in pts:
            if flip_x:
                x_pct = 100.0 - float(x_pct)

            x = float(x_pct) / 100.0 * w
            y = float(y_pct) / 100.0 * h
            xy.append((x, y))

        if len(xy) >= 3:
            draw.polygon(xy, fill=255)

    return mask


# =========================
# MODEL
# Must match training model exactly
# =========================
class SimpleCNN_MultiHead(nn.Module):
    def __init__(self, in_channels=2):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 16, 3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(128, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(256, 512, 3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )

        self.gap = nn.AdaptiveAvgPool2d((1, 1))

        self.shared_fc = nn.Sequential(
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
        )

        self.spatial_head = nn.Linear(512, 2)
        self.threat_head = nn.Linear(512, 2)

    def forward(self, x):
        x = self.features(x)
        x = self.gap(x)
        x = x.view(x.size(0), -1)
        x = self.shared_fc(x)

        spatial_logits = self.spatial_head(x)
        threat_logits = self.threat_head(x)

        return spatial_logits, threat_logits


# =========================
# DATA HELPERS
# =========================
def read_index_csv(path):
    rows = []
    with open(path, "r", newline="") as f:
        reader = csv.DictReader(f)
        rows.extend(reader)

    if not rows:
        raise RuntimeError(f"index.csv is empty: {path}")

    return rows


def load_label_map(label_path):
    data = json.load(open(label_path, "r"))

    label_map = {}
    bad = []

    for item in data:
        image_path = item.get("image", "").replace("\\", "/").strip()
        if not image_path:
            bad.append(("missing image", item))
            continue

        fname = Path(image_path).name

        overlap = int(item.get("overlap", 0))
        isolated = int(item.get("isolated", 0))
        contraband = int(item.get("contraband", 0))
        non_contraband = int(item.get("non_contraband", 0))

        # spatial: 0=isolated, 1=overlap
        if overlap == 1 and isolated == 0:
            spatial = 1
        elif overlap == 0 and isolated == 1:
            spatial = 0
        else:
            bad.append((image_path, f"bad spatial: overlap={overlap}, isolated={isolated}"))
            continue

        # threat: 0=non_contraband, 1=contraband
        if contraband == 1 and non_contraband == 0:
            threat = 1
        elif contraband == 0 and non_contraband == 1:
            threat = 0
        else:
            bad.append((image_path, f"bad threat: contraband={contraband}, non_contraband={non_contraband}"))
            continue

        labels = {
            "spatial": spatial,
            "threat": threat,
        }

        label_map[image_path] = labels
        label_map[fname] = labels

    if bad:
        print("Bad label examples:")
        for x in bad[:20]:
            print(x)
        raise ValueError(f"Found {len(bad)} bad label rows")

    print(f"Loaded labels: {len(data)} rows from {label_path}")
    return label_map


def resize_keep_ratio_and_pad(pil_img, target_size, interpolation, fill=255):
    """
    Same preprocessing style as training validation.

    Resize the image to fit inside target_size x target_size while keeping
    aspect ratio, then pad to exactly target_size x target_size.

    This avoids the zoom-in problem caused by resize + center_crop.

    For X-ray image: fill=255 gives white padding.
    For item mask:   fill=0 gives blank mask padding.
    """
    img = pil_img.convert("L")
    w, h = img.size

    scale = min(target_size / w, target_size / h)
    new_w = int(round(w * scale))
    new_h = int(round(h * scale))

    img = TF.resize(
        img,
        [new_h, new_w],
        interpolation=interpolation,
    )

    pad_left = (target_size - new_w) // 2
    pad_top = (target_size - new_h) // 2
    pad_right = target_size - new_w - pad_left
    pad_bottom = target_size - new_h - pad_top

    img = TF.pad(
        img,
        padding=[pad_left, pad_top, pad_right, pad_bottom],
        fill=fill,
    )

    return img


class JointEvalTransform:
    """
    Evaluation transform that matches training validation preprocessing:
    - keep full image visible
    - keep aspect ratio
    - pad to 512 x 512
    - no zoom
    - no crop
    - no random augmentation
    """
    def __init__(self, image_size):
        self.image_size = image_size

    def __call__(self, img, item_mask):
        img = resize_keep_ratio_and_pad(
            img,
            target_size=self.image_size,
            interpolation=TF.InterpolationMode.BILINEAR,
            fill=255,
        )

        item_mask = resize_keep_ratio_and_pad(
            item_mask,
            target_size=self.image_size,
            interpolation=TF.InterpolationMode.NEAREST,
            fill=0,
        )

        img_t = TF.to_tensor(img)
        mask_t = TF.to_tensor(item_mask)

        img_t = TF.normalize(img_t, GRAY_MEAN, GRAY_STD)
        mask_t = TF.normalize(mask_t, MASK_MEAN, MASK_STD)

        x = torch.cat([img_t, mask_t], dim=0)
        return x


class MultiHeadItemMaskEvalDataset(Dataset):
    def __init__(self, index_rows, processed_root, split, label_map, transform=None):
        self.processed_root = processed_root
        self.split = split.lower()
        self.transform = transform

        self.filepaths = []
        self.spatial_labels = []
        self.threat_labels = []
        self.has_item_polygon = []

        missing = []

        for r in index_rows:
            fp = (r.get("filepath") or "").replace("\\", "/").strip()
            row_split = (r.get("split") or "").strip().lower()

            if row_split != self.split:
                continue

            if not fp:
                missing.append("missing filepath")
                continue

            fname = Path(fp).name
            label = label_map.get(fp, label_map.get(fname, None))

            if label is None:
                missing.append(f"missing label for {fp}")
                continue

            img_path = self.processed_root / fp

            if not img_path.exists():
                missing.append(f"missing image file: {img_path}")
                continue

            polygon_key = find_original_polygon_key(fname)

            self.filepaths.append(fp)
            self.spatial_labels.append(int(label["spatial"]))
            self.threat_labels.append(int(label["threat"]))
            self.has_item_polygon.append(polygon_key is not None)

        if missing:
            raise RuntimeError(f"Missing labels/files: {len(missing)} examples. First few: {missing[:10]}")

        print(f"Loaded {len(self.filepaths)} {split} samples")
        print(f"{sum(self.has_item_polygon)} samples have item polygons")
        print(f"{len(self.has_item_polygon) - sum(self.has_item_polygon)} samples have blank item mask")

    def __len__(self):
        return len(self.filepaths)

    def __getitem__(self, idx):
        img_path = self.processed_root / self.filepaths[idx]

        img = Image.open(img_path).convert("L")

        item_mask = create_item_only_mask(
            processed_filename=Path(self.filepaths[idx]).name,
            image_size=img.size,
        )

        if self.transform:
            x = self.transform(img, item_mask)
        else:
            img_t = TF.normalize(TF.to_tensor(img), GRAY_MEAN, GRAY_STD)
            mask_t = TF.normalize(TF.to_tensor(item_mask), MASK_MEAN, MASK_STD)
            x = torch.cat([img_t, mask_t], dim=0)

        spatial_y = torch.tensor(self.spatial_labels[idx], dtype=torch.long)
        threat_y = torch.tensor(self.threat_labels[idx], dtype=torch.long)

        return x, spatial_y, threat_y, self.filepaths[idx]


# =========================
# LOAD DATA
# =========================
index_rows = read_index_csv(INDEX_CSV)
label_map = load_label_map(TEST_LABELS_JSON)

eval_transform = JointEvalTransform(IMAGE_SIZE)

test_ds = MultiHeadItemMaskEvalDataset(
    index_rows=index_rows,
    processed_root=PROCESSED_ROOT,
    split="test",
    label_map=label_map,
    transform=eval_transform,
)

test_loader = DataLoader(
    test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

# Quick shape check
x, spatial_y, threat_y, paths = next(iter(test_loader))
print("Batch input shape:", x.shape)  # should be [B, 2, 512, 512]
print("Batch spatial shape:", spatial_y.shape)
print("Batch threat shape:", threat_y.shape)

# =========================
# LOAD MODEL SAFELY
# =========================
model = SimpleCNN_MultiHead(in_channels=2).to(device)

try:
    state = torch.load(MODEL_PATH, map_location=device, weights_only=True)
except TypeError:
    state = torch.load(MODEL_PATH, map_location=device)

# Supports either raw state_dict or checkpoint dict
if isinstance(state, dict):
    if "model_state" in state:
        state = state["model_state"]
    elif "model_state_dict" in state:
        state = state["model_state_dict"]

model.load_state_dict(state, strict=True)
model.eval()

print("Loaded model:", MODEL_PATH)

# =========================
# EVALUATE
# =========================
all_paths = []

spatial_true = []
spatial_pred = []
spatial_prob = []

threat_true = []
threat_pred = []
threat_prob = []

with torch.no_grad():
    for imgs, spatial_y, threat_y, paths in test_loader:
        imgs = imgs.to(device, non_blocking=True)

        spatial_logits, threat_logits = model(imgs)

        spatial_probs = torch.softmax(spatial_logits, dim=1)
        threat_probs = torch.softmax(threat_logits, dim=1)

        spatial_preds = spatial_probs.argmax(dim=1)
        threat_preds = threat_probs.argmax(dim=1)

        spatial_true.extend(spatial_y.cpu().numpy().tolist())
        spatial_pred.extend(spatial_preds.cpu().numpy().tolist())
        spatial_prob.extend(spatial_probs.cpu().numpy().tolist())

        threat_true.extend(threat_y.cpu().numpy().tolist())
        threat_pred.extend(threat_preds.cpu().numpy().tolist())
        threat_prob.extend(threat_probs.cpu().numpy().tolist())

        all_paths.extend(paths)

spatial_true = np.array(spatial_true)
spatial_pred = np.array(spatial_pred)
spatial_prob = np.array(spatial_prob)

threat_true = np.array(threat_true)
threat_pred = np.array(threat_pred)
threat_prob = np.array(threat_prob)

both_correct = (spatial_true == spatial_pred) & (threat_true == threat_pred)

# =========================
# METRICS
# =========================
def compute_head_metrics(name, y_true, y_pred, y_prob, class_names):
    acc = float((y_true == y_pred).mean())

    p, r, f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="binary",
        zero_division=0,
    )

    try:
        auc = roc_auc_score(y_true, y_prob[:, 1])
    except Exception:
        auc = 0.0

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])

    print(f"\n===== {name.upper()} HEAD =====")
    print(f"Class 0: {class_names[0]}")
    print(f"Class 1: {class_names[1]}")
    print(f"Accuracy : {acc:.4f}")
    print(f"Precision: {p:.4f}")
    print(f"Recall   : {r:.4f}")
    print(f"F1       : {f1:.4f}")
    print(f"AUC      : {auc:.4f}")
    print("Confusion Matrix rows=true, cols=pred:")
    print(cm)

    print("\nClassification Report:")
    print(classification_report(
        y_true,
        y_pred,
        target_names=class_names,
        zero_division=0,
    ))

    return {
        "accuracy": acc,
        "precision": float(p),
        "recall": float(r),
        "f1": float(f1),
        "auc": float(auc),
        "cm": cm.tolist(),
    }

print("\n===== OVERALL RESULT =====")
print(f"Total samples : {len(all_paths)}")
print(f"Both correct  : {int(both_correct.sum())}")
print(f"Both wrong    : {int((~both_correct).sum())}")
print(f"Both accuracy : {float(both_correct.mean()):.4f}")

spatial_metrics = compute_head_metrics(
    "spatial",
    spatial_true,
    spatial_pred,
    spatial_prob,
    SPATIAL_CLASSES,
)

threat_metrics = compute_head_metrics(
    "threat",
    threat_true,
    threat_pred,
    threat_prob,
    THREAT_CLASSES,
)

# =========================
# RESULTS DATAFRAME
# =========================
results_df = pd.DataFrame({
    "image": all_paths,

    "spatial_true_id": spatial_true,
    "spatial_pred_id": spatial_pred,
    "spatial_true": [SPATIAL_CLASSES[i] for i in spatial_true],
    "spatial_pred": [SPATIAL_CLASSES[i] for i in spatial_pred],
    "spatial_prob_isolated": spatial_prob[:, 0],
    "spatial_prob_overlap": spatial_prob[:, 1],
    "spatial_correct": spatial_true == spatial_pred,

    "threat_true_id": threat_true,
    "threat_pred_id": threat_pred,
    "threat_true": [THREAT_CLASSES[i] for i in threat_true],
    "threat_pred": [THREAT_CLASSES[i] for i in threat_pred],
    "threat_prob_non_contraband": threat_prob[:, 0],
    "threat_prob_contraband": threat_prob[:, 1],
    "threat_correct": threat_true == threat_pred,

    "both_correct": both_correct,
})

results_csv = OUT_DIR / "test_predictions.csv"
results_df.to_csv(results_csv, index=False)

metrics_json = OUT_DIR / "test_metrics.json"
with open(metrics_json, "w") as f:
    json.dump({
        "model_path": str(MODEL_PATH),
        "total_samples": int(len(all_paths)),
        "both_correct": int(both_correct.sum()),
        "both_wrong": int((~both_correct).sum()),
        "both_accuracy": float(both_correct.mean()),
        "spatial_metrics": spatial_metrics,
        "threat_metrics": threat_metrics,
        "spatial_classes": SPATIAL_CLASSES,
        "threat_classes": THREAT_CLASSES,
        "input_channels": ["grayscale_image", "item_only_mask"],
        "item_mask_labels": ["shampoo", "blade"],
        "excluded_mask_labels": ["tray"],
        "image_size": IMAGE_SIZE,
        "gray_mean": GRAY_MEAN,
        "gray_std": GRAY_STD,
        "mask_mean": MASK_MEAN,
        "mask_std": MASK_STD,
    }, f, indent=2)

print("\nSaved predictions to:", results_csv)
print("Saved metrics to:", metrics_json)

correct_indices = np.where(both_correct)[0]
wrong_indices = np.where(~both_correct)[0]

print("\n===== SAMPLE PREDICTIONS =====")
display(results_df.head(10))

print("\nCorrect examples:", correct_indices[:10])
print("Wrong examples:", wrong_indices[:10])

# =========================
# VISUALIZE IMAGE + ITEM MASK + PREDICTION
# =========================
def show_prediction(idx):
    row = results_df.iloc[int(idx)]

    img_path = PROCESSED_ROOT / row["image"]

    original_img = Image.open(img_path).convert("L")
    original_item_mask = create_item_only_mask(
        processed_filename=Path(row["image"]).name,
        image_size=original_img.size,
    )

    # Show the exact 512x512 padded input that the model receives.
    x_vis = eval_transform(original_img, original_item_mask)
    model_img = x_vis[0].cpu().numpy() * GRAY_STD[0] + GRAY_MEAN[0]
    model_img = np.clip(model_img, 0, 1)
    model_mask = x_vis[1].cpu().numpy() * MASK_STD[0] + MASK_MEAN[0]
    model_mask = np.clip(model_mask, 0, 1)

    plt.figure(figsize=(18, 5))

    plt.subplot(1, 4, 1)
    plt.imshow(original_img, cmap="gray")
    plt.title(f"Original image\n{original_img.size[0]}x{original_img.size[1]}")
    plt.axis("off")

    plt.subplot(1, 4, 2)
    plt.imshow(original_item_mask, cmap="gray")
    plt.title("Original item mask")
    plt.axis("off")

    plt.subplot(1, 4, 3)
    plt.imshow(model_img, cmap="gray", vmin=0, vmax=1)
    plt.title(f"Model input image\n{IMAGE_SIZE}x{IMAGE_SIZE}, padded no crop")
    plt.axis("off")

    plt.subplot(1, 4, 4)
    plt.imshow(model_img, cmap="gray", vmin=0, vmax=1)
    plt.imshow(model_mask, alpha=0.35, cmap="Reds", vmin=0, vmax=1)
    plt.title(
        f"Spatial: {row['spatial_true']} → {row['spatial_pred']}\n"
        f"Threat: {row['threat_true']} → {row['threat_pred']}\n"
        f"Both correct: {row['both_correct']}"
    )
    plt.axis("off")

    plt.tight_layout()
    plt.show()

print("\nShowing one sample prediction:")
show_prediction(0)

if len(wrong_indices) > 0:
    print("\nShowing one wrong prediction:")
    show_prediction(wrong_indices[0])
else:
    print("\nNo wrong predictions found.")

# =========================
# GRAD-CAM FOR MULTI-HEAD MODEL
# =========================
class GradCAM:
    def __init__(self, model, target_layer, head_name="threat"):
        self.model = model
        self.target_layer = target_layer
        self.head_name = head_name

        self.gradients = None
        self.activations = None

        self.fwd_handle = target_layer.register_forward_hook(self.forward_hook)
        self.bwd_handle = target_layer.register_full_backward_hook(self.backward_hook)

    def forward_hook(self, module, input, output):
        self.activations = output.detach()

    def backward_hook(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach()

    def generate(self, input_tensor, target_class):
        self.model.zero_grad(set_to_none=True)

        spatial_logits, threat_logits = self.model(input_tensor)

        if self.head_name == "spatial":
            logits = spatial_logits
        elif self.head_name == "threat":
            logits = threat_logits
        else:
            raise ValueError("head_name must be 'spatial' or 'threat'")

        score = logits[:, target_class].sum()
        score.backward()

        gradients = self.gradients
        activations = self.activations

        weights = gradients.mean(dim=(2, 3), keepdim=True)
        cam = (weights * activations).sum(dim=1, keepdim=True)
        cam = torch.relu(cam)

        cam = torch.nn.functional.interpolate(
            cam,
            size=input_tensor.shape[2:],
            mode="bilinear",
            align_corners=False,
        )

        cam = cam.squeeze().cpu().numpy()
        cam = cam - cam.min()
        cam = cam / (cam.max() + 1e-8)

        return cam

    def remove_hooks(self):
        self.fwd_handle.remove()
        self.bwd_handle.remove()


def denormalize_gray(tensor_img_channel):
    img = tensor_img_channel.cpu().numpy()
    img = img * GRAY_STD[0] + GRAY_MEAN[0]
    img = np.clip(img, 0, 1)
    return img


def denormalize_mask(tensor_mask_channel):
    mask = tensor_mask_channel.cpu().numpy()
    mask = mask * MASK_STD[0] + MASK_MEAN[0]
    mask = np.clip(mask, 0, 1)
    return mask


def overlay_gradcam(gray_img, cam):
    gray_uint8 = np.uint8(gray_img * 255)
    heatmap = np.uint8(255 * cam)

    heatmap_color = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)
    gray_color = cv2.cvtColor(gray_uint8, cv2.COLOR_GRAY2BGR)

    overlay = cv2.addWeighted(gray_color, 0.55, heatmap_color, 0.45, 0)
    overlay = cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB)

    return overlay


def get_conv_layers(model):
    conv_layers = []

    for idx, layer in enumerate(model.features):
        if isinstance(layer, nn.Conv2d):
            conv_layers.append((f"features[{idx}]", layer))

    return conv_layers


conv_layers = get_conv_layers(model)

print("\nDetected Conv layers:")
for name, layer in conv_layers:
    print(name, layer)


def show_gradcam_for_example(dataset, index=0, head_name="threat", target_class=1, layer_index=-1):
    """
    head_name:
      "spatial" target_class 0=isolated, 1=overlap
      "threat"  target_class 0=non_contraband, 1=contraband
    """

    model.eval()

    x, spatial_y, threat_y, rel_path = dataset[int(index)]
    input_tensor = x.unsqueeze(0).to(device)

    with torch.no_grad():
        spatial_logits, threat_logits = model(input_tensor)
        spatial_probs = torch.softmax(spatial_logits, dim=1).cpu().numpy()[0]
        threat_probs = torch.softmax(threat_logits, dim=1).cpu().numpy()[0]

        spatial_pred = int(np.argmax(spatial_probs))
        threat_pred = int(np.argmax(threat_probs))

    gray_img = denormalize_gray(x[0])
    item_mask_img = denormalize_mask(x[1])

    layer_name, layer = conv_layers[layer_index]

    gradcam = GradCAM(model, layer, head_name=head_name)
    cam = gradcam.generate(input_tensor, target_class=target_class)
    gradcam.remove_hooks()

    overlay = overlay_gradcam(gray_img, cam)

    print("\nImage:", rel_path)
    print("Grad-CAM head:", head_name)
    print("Grad-CAM target class:", target_class)
    print("Grad-CAM layer:", layer_name)

    print("\nTrue spatial:", SPATIAL_CLASSES[int(spatial_y)])
    print("Pred spatial:", SPATIAL_CLASSES[spatial_pred])
    print("Spatial probs:")
    for i, name in enumerate(SPATIAL_CLASSES):
        print(f"  {name}: {spatial_probs[i]:.4f}")

    print("\nTrue threat:", THREAT_CLASSES[int(threat_y)])
    print("Pred threat:", THREAT_CLASSES[threat_pred])
    print("Threat probs:")
    for i, name in enumerate(THREAT_CLASSES):
        print(f"  {name}: {threat_probs[i]:.4f}")

    plt.figure(figsize=(18, 5))

    plt.subplot(1, 4, 1)
    plt.imshow(gray_img, cmap="gray", vmin=0, vmax=1)
    plt.title("Input image")
    plt.axis("off")

    plt.subplot(1, 4, 2)
    plt.imshow(item_mask_img, cmap="gray", vmin=0, vmax=1)
    plt.title("Item-only mask")
    plt.axis("off")

    plt.subplot(1, 4, 3)
    plt.imshow(gray_img, cmap="gray", vmin=0, vmax=1)
    plt.imshow(item_mask_img, alpha=0.35, cmap="Reds")
    plt.title("Image + item mask")
    plt.axis("off")

    plt.subplot(1, 4, 4)
    plt.imshow(overlay)
    plt.title(f"Grad-CAM: {head_name}, class {target_class}")
    plt.axis("off")

    plt.tight_layout()
    plt.show()


def show_gradcam_correct_and_wrong(max_images=3):
    print("\nGrad-CAM on correct examples:")

    for idx in correct_indices[:max_images]:
        show_gradcam_for_example(
            dataset=test_ds,
            index=int(idx),
            head_name="threat",
            target_class=1,
            layer_index=-1,
        )

    print("\nGrad-CAM on wrong examples:")

    if len(wrong_indices) == 0:
        print("No wrong examples found.")
        return

    for idx in wrong_indices[:max_images]:
        show_gradcam_for_example(
            dataset=test_ds,
            index=int(idx),
            head_name="threat",
            target_class=1,
            layer_index=-1,
        )


# Run Grad-CAM examples
show_gradcam_correct_and_wrong(max_images=3)